# 02 — Feature engineering, line by line

This notebook deliberately implements each feature and preprocessing step directly. Nothing is imported from `firmaware.features` or `firmaware.transform`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "deployment_events.csv"

ID_COLUMNS = ["deployment_id", "device_id", "site_id", "firmware_fingerprint"]
LEAKAGE_COLUMNS = ["time_to_failure_hours", "rollback_required"]
CATEGORICAL_COLUMNS = ["vendor_name", "device_type", "hardware_series", "fleet_tier", "site_criticality", "deployment_type"]
NUMERIC_COLUMNS = [
    "version_jump_magnitude", "kernel_touched", "bootloader_touched",
    "protocol_mismatch_flag", "maintenance_window", "network_stress_score",
    "error_rate_predeploy", "uptime_days", "past_failure_count",
    "firmware_release_age_days", "cve_count", "max_cvss_score",
    "cross_vendor_dependency_count", "dependent_device_count",
]
DERIVED_NUMERIC_COLUMNS = ["major_version_jump", "major_version_changed", "emergency_no_maintenance", "core_system_touched"]
DATE_COLUMN = "deployment_date"
OUTCOME_COLUMN = "deployment_outcome"
LABEL_COLUMN = "deployment_risk"
ALLOWED_OUTCOMES = {"SUCCESS", "DEGRADED", "ROLLBACK", "FAILED"}

raw = pd.read_csv(DATA_PATH)
assert set(raw[OUTCOME_COLUMN].dropna()).issubset(ALLOWED_OUTCOMES)
assert not raw["deployment_id"].duplicated().any()
raw[DATE_COLUMN] = pd.to_datetime(raw[DATE_COLUMN], errors="coerce")
assert raw[DATE_COLUMN].notna().all()
for column in NUMERIC_COLUMNS:
    before = raw[column].notna()
    raw[column] = pd.to_numeric(raw[column], errors="coerce")
    assert not (before & raw[column].isna()).any(), column
print(raw.shape)

(20000, 30)


## Explicit derivations

`major_version_jump` remains signed so downgrades are visible. The label is vocabulary-independent after the allowed-outcome check.

In [2]:
def parse_major(value):
    if pd.isna(value):
        return np.nan
    try:
        return int(str(value).split(".", 1)[0])
    except (TypeError, ValueError):
        return np.nan

current_major = raw["current_firmware"].map(parse_major)
target_major = raw["target_firmware"].map(parse_major)
parse_failures = current_major.isna() | target_major.isna()
assert parse_failures.mean() <= 0.01, f"Version parse failure rate: {parse_failures.mean():.2%}"

featured = raw.copy()
featured["major_version_jump"] = target_major - current_major
featured["major_version_changed"] = np.where(
    featured["major_version_jump"].isna(),
    np.nan,
    featured["major_version_jump"].ne(0).astype(int),
)
featured["emergency_no_maintenance"] = (
    featured["deployment_type"].eq("EMERGENCY") & featured["maintenance_window"].eq(0)
).astype(int)
featured["core_system_touched"] = (
    featured["kernel_touched"].fillna(0).ne(0) | featured["bootloader_touched"].fillna(0).ne(0)
).astype(int)
featured[LABEL_COLUMN] = featured[OUTCOME_COLUMN].ne("SUCCESS").astype(int)

deployment_ids = featured["deployment_id"].copy()
featured = featured.drop(columns=ID_COLUMNS + LEAKAGE_COLUMNS + ["current_firmware", "target_firmware"])
featured.index = pd.Index(deployment_ids, name="deployment_id")

display(featured[DERIVED_NUMERIC_COLUMNS + [OUTCOME_COLUMN, LABEL_COLUMN]].head())

,major_version_jump,major_version_changed,emergency_no_maintenance,core_system_touched,deployment_outcome,deployment_risk
deployment_id,,,,,,
DEP_000001,0,0.0,0,0,ROLLBACK,1
DEP_000002,2,1.0,0,1,ROLLBACK,1
DEP_000003,1,1.0,0,0,FAILED,1
DEP_000004,2,1.0,1,1,FAILED,1
DEP_000005,0,0.0,0,0,DEGRADED,1


In [3]:
feature_summary = pd.DataFrame({
    "missing": featured[DERIVED_NUMERIC_COLUMNS].isna().sum(),
    "mean": featured[DERIVED_NUMERIC_COLUMNS].mean(),
    "minimum": featured[DERIVED_NUMERIC_COLUMNS].min(),
    "maximum": featured[DERIVED_NUMERIC_COLUMNS].max(),
})
display(feature_summary)
display(featured["major_version_jump"].value_counts(dropna=False).sort_index().to_frame("rows"))

,missing,mean,minimum,maximum
major_version_jump,0,0.30090,0.0,2.0
major_version_changed,0,0.20145,0.0,1.0
emergency_no_maintenance,0,0.12200,0.0,1.0
core_system_touched,0,0.18785,0.0,1.0


,rows
major_version_jump,
0,15971
1,2040
2,1989


## Strict chronological split

The date at the 80% position starts the test side. All rows on that date move together, ensuring every train date is earlier than every test date.

In [4]:
ordered = featured.sort_values(DATE_COLUMN, kind="mergesort")
position = min(max(int(np.floor(len(ordered) * 0.8)), 1), len(ordered) - 1)
cutoff = ordered[DATE_COLUMN].iloc[position]
train_side = ordered.loc[ordered[DATE_COLUMN] < cutoff].copy()
test_side = ordered.loc[ordered[DATE_COLUMN] >= cutoff].copy()
assert not train_side.empty and not test_side.empty
assert train_side[DATE_COLUMN].max() < test_side[DATE_COLUMN].min()
pd.Series({
    "cutoff": cutoff,
    "train_rows": len(train_side),
    "test_rows": len(test_side),
    "train_end": train_side[DATE_COLUMN].max(),
    "test_start": test_side[DATE_COLUMN].min(),
})

cutoff        2023-01-23 00:00:00
train_rows                  15987
test_rows                    4013
train_end     2023-01-22 00:00:00
test_start    2023-01-23 00:00:00
dtype: object

## Preprocessing, fitted on training rows only

The order is **impute → encode → align → scale**. Numeric medians, category vocabulary, and scaling statistics are all learned only from the chronological training side.

In [5]:
MODEL_DROP_COLUMNS = [DATE_COLUMN, OUTCOME_COLUMN, LABEL_COLUMN]
X_train_raw = train_side.drop(columns=MODEL_DROP_COLUMNS)
X_test_raw = test_side.drop(columns=MODEL_DROP_COLUMNS)
y_train = train_side[LABEL_COLUMN].astype(int)
y_test = test_side[LABEL_COLUMN].astype(int)
model_numeric_columns = NUMERIC_COLUMNS + DERIVED_NUMERIC_COLUMNS

# 1. Impute using train medians.
train_numeric = X_train_raw[model_numeric_columns].astype(float)
test_numeric = X_test_raw[model_numeric_columns].astype(float)
medians = train_numeric.median()
assert medians.notna().all(), "A numeric feature is entirely missing in train"
train_numeric = train_numeric.fillna(medians)
test_numeric = test_numeric.fillna(medians)

# 2. One-hot encode using the train category vocabulary.
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_encoded_values = encoder.fit_transform(X_train_raw[CATEGORICAL_COLUMNS])
test_encoded_values = encoder.transform(X_test_raw[CATEGORICAL_COLUMNS])
encoded_names = encoder.get_feature_names_out(CATEGORICAL_COLUMNS).tolist()
train_encoded = pd.DataFrame(train_encoded_values, columns=encoded_names, index=X_train_raw.index)
test_encoded = pd.DataFrame(test_encoded_values, columns=encoded_names, index=X_test_raw.index)

# 3. Align both matrices to the fitted feature list in raw space.
feature_list = model_numeric_columns + encoded_names
X_train = pd.concat([train_numeric, train_encoded], axis=1).reindex(columns=feature_list, fill_value=0.0)
X_test = pd.concat([test_numeric, test_encoded], axis=1).reindex(columns=feature_list, fill_value=0.0)

# 4. Scale numeric columns only; one-hot columns stay 0/1.
scaler = StandardScaler()
X_train.loc[:, model_numeric_columns] = scaler.fit_transform(X_train[model_numeric_columns])
X_test.loc[:, model_numeric_columns] = scaler.transform(X_test[model_numeric_columns])
X_train = X_train.astype(float)
X_test = X_test.astype(float)

print(f"train matrix={X_train.shape}; test matrix={X_test.shape}; positive train share={y_train.mean():.3f}")

train matrix=(15987, 85); test matrix=(4013, 85); positive train share=0.459


## Leakage assertion and unseen-category trace

In [6]:
forbidden = set(ID_COLUMNS + LEAKAGE_COLUMNS + [DATE_COLUMN, OUTCOME_COLUMN, "current_firmware", "target_firmware"])
assert forbidden.isdisjoint(feature_list)

probe = X_test_raw.iloc[[0]].copy()
probe.loc[:, "vendor_name"] = "Moxa"
unseen = {}
for index, column in enumerate(CATEGORICAL_COLUMNS):
    known = set(encoder.categories_[index])
    value = probe.iloc[0][column]
    if value not in known:
        unseen[column] = value

probe_encoded = encoder.transform(probe[CATEGORICAL_COLUMNS])
probe_encoded = pd.DataFrame(probe_encoded, columns=encoded_names)
vendor_block = [column for column in encoded_names if column.startswith("vendor_name_")]
assert unseen == {"vendor_name": "Moxa"}
assert probe_encoded[vendor_block].sum(axis=1).iloc[0] == 0.0
print("unseen report:", unseen)
print("vendor one-hot block sum:", probe_encoded[vendor_block].sum(axis=1).iloc[0])

unseen report: {'vendor_name': 'Moxa'}
vendor one-hot block sum: 0.0
